<h2>Upload video to youtube</h2>

In [1]:
import os
import pickle

from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

/opt/homebrew/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]


In [5]:
#authorize to save permanent token
def get_authenticated_service():
    creds = None
    # load saved token
    if os.path.exists("martin_yt_token.json"):
        creds = Credentials.from_authorized_user_file("martin_yt_token.json", SCOPES)

    # if no valid credentials
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "secret_files/martin_yt_client_secret.json",
                SCOPES
            )
            creds = flow.run_local_server(port=0)
        # save token
        with open("martin_yt_token.json", "w") as token:
            token.write(creds.to_json())

    youtube = build("youtube", "v3", credentials=creds)
    return youtube

# youtube = get_authenticated_service() #permanent token is saved. Note: login with developer account (Tester type)



In [ ]:
def upload_video(file_path, title, description):
    request_body = {
        "snippet": {
            "title": title,
            "description": description,
            # "tags": tags,
            # "categoryId": "22"
        },
        "status": {
            "privacyStatus": "public"
        }
    }

    media = MediaFileUpload(file_path, resumable=True)

    request = youtube.videos().insert(
        part="snippet,status",
        body=request_body,
        media_body=media
    )

    response = None

    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"Uploading {int(status.progress() * 100)}%")

    print("Upload complete")
    print("Video ID:", response["id"])

#test
file_path = '/Users/sangdo/Downloads/math_games_video/output/9.mp4'
title = 'Math games for your kids at the spare time - Puzzle 9'
description = 'We introduce a range of various games:\n \
Addition matrix\n \
Hidden gems\n \
Word search\n \
Crossword numbers\n \
Balance game\n \
Find lines\n \
Triangle sum\n \
Balance fruit\n \
Object coordination\n \
Spy game\n \
Detect shape\n \
Bee house\n \
\n \
The link to download more games as PDF file in the first comment.'

# upload_video(file_path, title, description)